# 07 — Engine-A: per-token operation-cycle go/no-go gate

**ENGINE_A_DESIGN §5.** operation-축 corpus(Super-NaturalInstructions)로 per-token reconstruction cycle 학습 → QuAIL probe 에서 sequence code(`mean_t α`) 추출 → Hewitt-Liang selectivity.

## Verdict 기준
- **PASS (go)**: acc(operation) 가 4 control {random / topic / token-type / geometry} 전부 대비 유의미 ↑ (특히 topic·geometry). → Engine-B 진입.
- **FAIL**: operation 분리 안 됨 → 방향 재고. (per-token 도 topic으로 무너지면 pivot 자체 재검토.)

LLM-free. 코드/loss/eval/runner 는 모두 CPU 테스트 통과(`tests/test_opcycle_*.py`, `test_engine_a.py`). 여기서는 실데이터로 판정만.

In [ ]:
import os, sys
BASE = '/content/drive/MyDrive/sideproject'
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
os.chdir(BASE)
if BASE not in sys.path:
    sys.path.insert(0, BASE)
import torch
print('cwd =', os.getcwd(), '| cuda =', torch.cuda.is_available())

## A. operation-축 학습 corpus (Super-NI) → per-token encode

topic-축 source (news/arxiv/tweet/amazon/reddit) 전부 off, `include_superni=True` 만. per-task cap=50 으로 operation 커버리지 넓게 (topic 만이 아닌 operation variance dominate).

In [ ]:
from phase1.data import CorpusConfig, build_corpus, encode_or_load_tokens

ccfg = CorpusConfig(
    include_news=False, include_arxiv=False, include_tweet=False,
    include_amazon=False, include_reddit=False,
    include_superni=True,                 # operation-axis training corpus
    superni_max_samples=40_000, superni_max_per_task=50,
)
corpus = build_corpus(ccfg)
train_tokens, train_mask = encode_or_load_tokens(corpus, t_cap=128, batch_size=64)
print('train tokens', train_tokens.shape, '| mask', train_mask.shape,
      '| sources', corpus['source'].value_counts().to_dict())

## B. QuAIL probe + 4 control 라벨

operation label = `question_type` (reasoning-type). topic = passage 평균 임베딩 KMeans 클러스터 (data-driven topic proxy). token-type = 길이 버킷 (surface-form artifact). random / geometry 는 runner 가 자동 생성.

In [ ]:
import numpy as np, pandas as pd
from sklearn.cluster import KMeans
from phase1.data import load_quail_probe

probe = load_quail_probe(ccfg).head(ccfg.quail_max_samples).reset_index(drop=True)
probe_tokens, probe_mask = encode_or_load_tokens(probe, t_cap=128, batch_size=64)

op_labels, op_uniques = pd.factorize(probe['label'])           # operation = question_type
print('probe', probe_tokens.shape, '| operations', len(op_uniques), list(op_uniques))

# topic control: KMeans on masked-mean passage embeddings (groups by content, not operation)
pooled = ((probe_tokens.astype('float32') * probe_mask[..., None]).sum(1)
          / probe_mask.sum(1, keepdims=True).clip(min=1))
topic = KMeans(n_clusters=min(8, len(op_uniques) + 2), n_init=10,
               random_state=0).fit_predict(pooled)
# token-type control: sequence-length bucket (pure surface form). QuAIL passages mostly hit
# the T=128 truncation cap, so quartile edges can collapse -> dedup them; selectivity_report
# auto-skips this control (with a warning) if it stays degenerate (<2 buckets).
lengths = probe_mask.sum(1)
edges = np.unique(np.quantile(lengths, [.25, .5, .75]))
token_type = np.digitize(lengths, edges)
controls = {'topic': topic}
if np.unique(token_type).size >= 2:
    controls['token_type'] = token_type
print('topic clusters', np.bincount(topic), '| token buckets', np.bincount(token_type),
      '| controls used:', list(controls))

## C. Engine-A 실행 → selectivity 판정

d_z=256 · K=16 · d_hidden=512 · ReMoE · per-token. K_active 붕괴(→0) 시 `lambdas={'lambda_lb': ...}` ↑ 또는 hard-concrete 전환(ENGINE_A_DESIGN §8).

In [ ]:
import json
from pathlib import Path
from phase1.engine_a import run_engine_a

result = run_engine_a(
    train_tokens, train_mask,
    probe_tokens, probe_mask, op_labels,
    control_label_sets=controls,
    d_z=256, k=16, d_hidden=512,
    epochs=40, batch_size=128,           # 128: per-step expert tensor stays GPU-safe
    lr=1e-3, seed=0,
    agg='meanmax',                       # mean+max: keep a few-token operation spike legible
    k_target=4.0,                        # ReMoE adaptive-L1 controller → mean K_active ≈ 4
    log_every=1, device='cuda' if torch.cuda.is_available() else 'cpu',
)
rep = result['report']
print('=' * 48)
print('VERDICT:', rep['verdict'])
print(f"acc(operation) = {rep['acc_operation']:.4f}  (chance-adj {rep['adj_operation']:.4f})")
for name, acc in rep['controls'].items():
    print(f"  control {name:11s} acc={acc:.4f}  selectivity(adj)={rep['selectivity'][name]:+.4f}")
for w in rep['warnings']:
    print('  ⚠', w)
print('mean K_active (last epoch):', round(result['history'][-1]['k_active'], 3),
      '| λ_l1', round(result['history'][-1]['lam_l1'], 5))
print('recon (first → last):', round(result['history'][0]['recon'], 4),
      '→', round(result['history'][-1]['recon'], 4))

out = Path('out/phase1/engine_a'); out.mkdir(parents=True, exist_ok=True)
(out / 'report.json').write_text(json.dumps({
    'verdict': rep['verdict'], 'acc_operation': rep['acc_operation'],
    'adj_operation': rep['adj_operation'], 'controls': rep['controls'],
    'adj_controls': rep['adj_controls'], 'selectivity': rep['selectivity'],
    'warnings': rep['warnings'], 'operations': list(map(str, op_uniques)),
    'history': result['history'],
}, indent=2))
print('saved →', out / 'report.json')

## D. 학습 곡선 (recon 수렴 + K_active sanity)

In [ ]:
import matplotlib.pyplot as plt
hist = result['history']
ep = range(1, len(hist) + 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(ep, [h['recon'] for h in hist]); ax[0].set_title('recon (1-cos)'); ax[0].set_xlabel('epoch')
ax[1].plot(ep, [h['k_active'] for h in hist]); ax[1].set_title('mean K_active'); ax[1].set_xlabel('epoch')
ax[1].axhline(1.0, ls='--', c='r', lw=.8, label='K=1 collapse'); ax[1].legend()
plt.tight_layout(); plt.show()

## E. 진단 사다리 — Engine-A FAIL 후 세 방향

첫 FAIL(acc(op)=0.10=chance, topic_sel=−0.65)은 **eval-입력 결함에 오염**됨: probe text가
`ctx + q + ans`(context 먼저)인데 BGE 우측 truncation(T=128)이 긴 QuAIL context로 채워져
operation을 담은 question·answer가 잘림 → 모델에 topic만 들어감. 아래는 세 실패 모드를 분리하는
진단 사다리.

| Stage 0b (raw-BGE ceiling) | Stage 1 (MoE gate) | 진단 | 다음 |
|---|---|---|---|
| operation 디코딩됨 | PASS | 존재 & 라우팅됨 | → Engine-B |
| operation 디코딩됨 | FAIL | 있는데 cycle이 topic으로 씻음(F3) | 방향 2 레버 |
| raw BGE에서도 안 됨 | (무의미) | BGE가 operation 미인코딩(F2) | 방향 3 encoder swap |

실행 전제: 위 A·B·C 셀이 먼저 실행되어 `ccfg`, `corpus`, `train_tokens`, `train_mask` 가 존재.

In [ ]:
# E0. Helper — QuAIL probe + operation/topic/token-type 라벨을 (question_first, t_cap) 별로 빌드.
import numpy as np, pandas as pd
from sklearn.cluster import KMeans
from phase1.data import load_quail_probe, encode_or_load_tokens

def build_probe(question_first, t_cap, encoder_name="BAAI/bge-large-en-v1.5"):
    p = (load_quail_probe(ccfg, question_first=question_first)
         .head(ccfg.quail_max_samples).reset_index(drop=True))
    ptok, pmask = encode_or_load_tokens(p, encoder_name=encoder_name, t_cap=t_cap, batch_size=64)
    op, op_u = pd.factorize(p['label'])                       # operation = question_type
    pooled = ((ptok.astype('float32') * pmask[..., None]).sum(1)
              / pmask.sum(1, keepdims=True).clip(min=1))
    topic = KMeans(n_clusters=min(8, len(op_u) + 2), n_init=10,
                   random_state=0).fit_predict(pooled)         # topic control
    edges = np.unique(np.quantile(pmask.sum(1), [.25, .5, .75]))
    tt = np.digitize(pmask.sum(1), edges)                      # token-type (length) control
    ctrls = {'topic': topic}
    if np.unique(tt).size >= 2:
        ctrls['token_type'] = tt
    return p, ptok, pmask, op, op_u, ctrls


### 방향 1 (F1) — Stage 0 ceiling check (NO MoE)

raw frozen-BGE 임베딩에 operation이 *선형적으로라도* 있는지. 0a(현 probe) vs 0b(question-first,
T=256) 비교 → truncation 결함 정량 + F2 여부 판정.

In [ ]:
# E1. Stage 0 — raw-BGE ceiling. adj_operation>0 & >geometry 면 operation 인코더에 존재.
from phase1.eval_opcycle import raw_embedding_report

def ceiling(question_first, t_cap, tag):
    _, ptok, pmask, op, op_u, ctrls = build_probe(question_first, t_cap)
    rep = raw_embedding_report(ptok, pmask, op, control_label_sets=ctrls, agg='meanmax', seed=0)
    print(f"[{tag}] acc(op)={rep['acc_operation']:.4f} adj={rep['adj_operation']:.4f} | "
          f"topic_sel={rep['selectivity'].get('topic', float('nan')):+.4f} "
          f"geom_sel={rep['selectivity'].get('geometry', float('nan')):+.4f} | ops={len(op_u)}")
    return rep

rep_0a = ceiling(False, 128, '0a current ctx-first T128')
rep_0b = ceiling(True,  256, '0b fixed   q-first  T256')
print('\nΔ adj_operation (0b − 0a):', round(rep_0b['adj_operation'] - rep_0a['adj_operation'], 4))
print('→ 0b adj>0 & >geom: operation 존재 → Stage 1(E2). 0b adj≈0: F2(encoder, E4).')


### 방향 1 (F1) — Stage 1 clean 재실행

probe만 수정 (question-first + T=256), sparsity 불변(k_target=4, epochs=40). verdict가 비로소
F3(cycle이 operation으로 라우팅하나)를 의미.

In [ ]:
# E2. 방향 1 — clean 재실행. (train_tokens/mask 는 셀 A 산출물)
import json
from pathlib import Path
from phase1.engine_a import run_engine_a

probe, probe_tokens, probe_mask, op_labels, op_uniques, controls = build_probe(True, 256)
result = run_engine_a(
    train_tokens, train_mask,
    probe_tokens, probe_mask, op_labels,
    control_label_sets=controls,
    d_z=256, k=16, d_hidden=512,
    epochs=40, batch_size=128, lr=1e-3, seed=0,
    agg='meanmax', k_target=4.0,
    log_every=1, device='cuda' if torch.cuda.is_available() else 'cpu',
)
rep = result['report']
print('VERDICT:', rep['verdict'],
      '| acc(op)', round(rep['acc_operation'], 4), 'adj', round(rep['adj_operation'], 4))
for n, a in rep['controls'].items():
    print(f"  {n:11s} acc={a:.4f} sel(adj)={rep['selectivity'][n]:+.4f}")
print('recon', round(result['history'][0]['recon'], 4), '→',
      round(result['history'][-1]['recon'], 4),
      '| K_active', round(result['history'][-1]['k_active'], 3))

out = Path('out/phase1/engine_a'); out.mkdir(parents=True, exist_ok=True)
(out / 'report_stage1.json').write_text(json.dumps({
    'verdict': rep['verdict'], 'acc_operation': rep['acc_operation'],
    'adj_operation': rep['adj_operation'], 'controls': rep['controls'],
    'selectivity': rep['selectivity'], 'warnings': rep['warnings'],
    'operations': list(map(str, op_uniques)), 'history': result['history'],
    'probe': 'question_first T256', 'ceiling_0b_adj': rep_0b['adj_operation'],
}, indent=2))
print('saved → out/phase1/engine_a/report_stage1.json')


### 방향 2 (F3) — operation은 있는데 cycle이 topic으로 씻어낼 때

Stage 1 FAIL & ceiling(0b) 높을 때만. (a) `use_best_recon` = over-sparsified 최종 대신 recon
최저 체크포인트, (b) `route_on_deviation` = sequence-global content(topic) 대신 within-sequence
국소 구조에 라우팅.

In [ ]:
# E3. 방향 2 — F3 레버. (E2의 probe_tokens/op_labels/controls 재사용)
for tag, kw in [('best_recon', dict(use_best_recon=True)),
                ('deviation',  dict(route_on_deviation=True)),
                ('both',       dict(use_best_recon=True, route_on_deviation=True))]:
    r = run_engine_a(
        train_tokens, train_mask, probe_tokens, probe_mask, op_labels,
        control_label_sets=controls, d_z=256, k=16, d_hidden=512,
        epochs=40, batch_size=128, lr=1e-3, seed=0, agg='meanmax', k_target=4.0,
        log_every=5, device='cuda' if torch.cuda.is_available() else 'cpu', **kw,
    )
    rr = r['report']
    print(f"[F3:{tag:10s}] {rr['verdict']} acc(op)={rr['acc_operation']:.4f} "
          f"adj={rr['adj_operation']:.4f} topic_sel={rr['selectivity'].get('topic', float('nan')):+.4f}")


### 방향 3 (F2) — frozen BGE가 operation 미인코딩일 때 encoder swap

Stage 0 0b의 adj≈0 이면 인코더 자체가 operation을 안 담는 것. `FrozenEncoder`는 임의 HF
`AutoModel` 지원이고 `run_engine_a`가 d_model 자동 추론 → `encoder_name`만 교체. 학습 전에 새
인코더로 **ceiling부터** 재측정해 operation이 잡히는지 확인.

In [ ]:
# E4. 방향 3 — encoder swap ceiling. adj가 BGE보다 유의미 높으면 그 인코더로 run_engine_a 진행.
ALT_ENCODER = 'intfloat/e5-large-v2'   # 후보: 'hkunlp/instructor-large', 'mixedbread-ai/mxbai-embed-large-v1'

# build_probe 재사용 → topic/token-type control 구성이 다른 셀과 동일하게 유지(드리프트 방지).
_, pt2, pm2, op2, opu2, ctrls2 = build_probe(True, 256, encoder_name=ALT_ENCODER)
rep2 = raw_embedding_report(pt2, pm2, op2, control_label_sets=ctrls2, agg='meanmax', seed=0)
print(f"[F2 ceiling {ALT_ENCODER}] acc(op)={rep2['acc_operation']:.4f} "
      f"adj={rep2['adj_operation']:.4f} topic_sel={rep2['selectivity'].get('topic', float('nan')):+.4f}")
print('→ BGE 0b adj 대비 ↑ 이면: tr2,trm2 = encode_or_load_tokens(corpus, encoder_name=ALT_ENCODER, t_cap=128)')
print('   후 run_engine_a(tr2, trm2, pt2, pm2, op2, control_label_sets=ctrls2, agg="meanmax", ...) 로 학습.')
